# Hemochromatosis Concept Expansion

This notebook expands from the haemochromatosis concept to find related terms using:
1. **SNOMED tree traversal** - Parent-child relationships via SNOMED CT
2. **MedCAT context similarity** - Vector-based semantic similarity

## Use Case

Given a starting SNOMED concept, find all related terms with their CUI codes for use in projects.

In [ ]:
# Add project root to path BEFORE importing packages
import os
import sys

CURRENT_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in locals()
    else os.getcwd()
)
project_root = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


# Set environment variables - allow override but provide sensible defaults
def find_snomed_rf2_path():
    env_path = os.environ.get("SCT2_PATH")
    if env_path and os.path.exists(env_path):
        return env_path
    import glob

    patterns = [
        os.path.join(project_root, "**/sct2_StatedRelationship_Full*.txt"),
        os.path.join(project_root, "uk_sct2cl_42.2.0/**/*StatedRelationship*INT*.txt"),
    ]
    for pattern in patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]
    # Fallback to relative path that user can override
    return os.path.join(
        project_root,
        "uk_sct2cl_42.2.0",
        "SnomedCT_InternationalRF2_*",
        "Full",
        "Terminology",
        "sct2_StatedRelationship_Full_INT_*.txt",
    )


os.environ["SNOMED_RF2_PATH"] = find_snomed_rf2_path()

# Default MedCAT model path
os.environ["MEDCAT_DHCAP02_PATH"] = os.environ.get(
    "MEDCAT_DHCAP02_PATH",
    os.path.join(project_root, "model_packs/medcat_model_pack_422d1d38fc58f158.zip"),
)


# Default UK SNOMED directory
def find_uk_snomed_dir():
    env_path = os.environ.get("UK_SNOMED_DIR")
    if env_path:
        return env_path
    import glob

    pattern = os.path.join(project_root, "uk_*/*/SnomedCT_UKClinicalRF2*")
    matches = glob.glob(pattern)
    if matches:
        return matches[0]
    return os.path.join(
        project_root, "uk_sct2cl_42.2.0", "SnomedCT_UKClinicalRF2_PRODUCTION_*"
    )


os.environ["UK_SNOMED_DIR"] = find_uk_snomed_dir()

## Setup

In [ ]:
import os
import sys

import pandas as pd

# Now import packages (sys.path should already be set from Cell 1)
from src.snomed_methods.snomed_methods_v1 import SnomedRelations

In [ ]:
# Initialize SnomedRelations WITH MedCAT enabled
snomed_relations = SnomedRelations(
    medcat=True, dhcap02=True
)  # Uses MEDCAT_DHCAP02_PATH environment variable

## Configuration Verification

In [ ]:
# Verify paths exist
critical_paths = [
    ("SNOMED RF2 Path", os.environ.get("SNOMED_RF2_PATH")),
    ("UK SNOMED_DIR", os.environ.get("UK_SNOMED_DIR")),
    ("MedCAT Model Pack", os.environ.get("MEDCAT_DHCAP02_PATH")),
]

for _path_name, path_value in critical_paths:
    exists = "exists" if os.path.exists(path_value) else "NOT FOUND"

## Define Starting Concepts

User-supplied list of concepts for expansion. Results will be combined from all concepts in the list.

Example usage:
```python
concepts_to_expand = [
    ("399187006", "haemochromatosis"),
    ("266549007", "iron overload"),
    ("198179005", "hemochromatosis syndrome"),
]
```

In [ ]:
# User-supplied list of concepts to expand
# Each tuple is (cui, term_name)
concepts_to_expand = [
    ("399187006", "haemochromatosis"),
    ("127606004", "Therapeutic phlebotomy"),
    ("56941009", "Hemochromatosis gene screening test"),
]

for _cui, _term in concepts_to_expand:
    pass

## Step 1: SNOMED Tree Expansion

Expand from each concept via parent-child relationships using recursive_code_expansion.

In [ ]:
# Perform tree expansion with n_recursion=5
n_recursion = 1

# Storage for all results
tree_results_list = []
all_tree_codes_str = set()
name_lookup_combined = {}


for starting_cui, starting_term in concepts_to_expand:
    # Convert CUI to string for consistency
    cui_str = str(starting_cui)
    codes, names = snomed_relations.recursive_code_expansion(
        cui_str, n_recursion=n_recursion, debug=False
    )

    tree_codes_set = set(codes) if codes else set()
    tree_names_list = names if names else []

    # Store individual concept results
    tree_results_list.append(
        {
            "original_term": starting_term,
            "codes": tree_codes_set,
            "names": tree_names_list,
            "count": len(tree_codes_set),
        }
    )

    # Add to combined set
    all_tree_codes_str.update(str(c) for c in tree_codes_set)

    # Build name lookup - convert codes to strings first
    for code, name in zip([str(c) for c in tree_codes_set], tree_names_list):
        if code not in name_lookup_combined:
            name_lookup_combined[code] = name

# Display summary
total_tree_count = sum(r["count"] for r in tree_results_list)

In [ ]:
# Display combined tree results
combined_tree_data = {
    "total_unique_concepts": len(all_tree_codes_str),
    "per_concept_results": [
        {"term": r["original_term"], "count": r["count"]} for r in tree_results_list
    ],
}
combined_tree_data

## Step 2: MedCAT Context Similarity

Find conceptually similar terms using MedCAT's context embeddings for each input concept.

In [ ]:
# Find similar concepts using MedCAT get_medcat_cdb_most_similar()
topn = 15

# Storage for all results
medcat_results_list = []
all_medcat_codes_str = set()


for starting_cui, starting_term in concepts_to_expand:
    # Convert CUI to string for consistency
    cui_str = str(starting_cui)
    medcat_codes, medcat_names = snomed_relations.get_medcat_cdb_most_similar(
        cui_str, context_type="xxxlong", type_id_filter=[], topn=topn
    )

    medcat_results_list.append(
        {
            "original_term": starting_term,
            "codes": medcat_codes,
            "names": medcat_names,
            "count": len(medcat_codes) if medcat_codes else 0,
        }
    )

    # Add to combined set
    all_medcat_codes_str.update(str(c) for c in medcat_codes)

# Display summary
total_medcat_count = sum(r["count"] for r in medcat_results_list)

In [ ]:
# Display combined MedCAT results
combined_medcat_data = {
    "total_unique_concepts": len(all_medcat_codes_str),
    "per_concept_results": [
        {"term": r["original_term"], "count": r["count"]} for r in medcat_results_list
    ],
}
combined_medcat_data

## Step 3: Combine Results

Merge tree expansion and MedCAT results from all concepts with MedCAT names taking precedence.

In [ ]:
# Build combined concept dictionary
# Merge all unique codes from both sources
all_codes_str = all_tree_codes_str.union(all_medcat_codes_str)

# Build combined concepts list with MedCAT names taking precedence
concepts_list = []

for code_str in sorted(all_codes_str):
    # Start with tree expansion name (may be None)
    name = name_lookup_combined.get(code_str, "N/A")

    # Check if this concept is in MedCAT results
    if code_str in all_medcat_codes_str:
        try:
            # Find the MedCAT name for this code across all concept results
            idx = None
            for result in medcat_results_list:
                for i, mc_code in enumerate(result["codes"]):
                    if str(mc_code) == code_str:
                        idx = i
                        break
                if idx is not None:
                    break

            # MedCAT names take precedence
            med_names = medcat_results_list[0]["names"] if idx is not None else []
            if idx is not None and med_names and idx < len(med_names):
                name = med_names[idx]
        except (ValueError, IndexError, TypeError):
            pass

    concepts_list.append(
        {"cui": code_str, "name": name if name else "N/A", "source": "combined"}
    )

# Build final results structure
all_concepts = {
    "combined_results": {
        "concepts": concepts_list,
        "total_count": len(concepts_list),
        "tree_count": len(all_tree_codes_str),
        "medcat_count": len(all_medcat_codes_str),
    }
}

# Display summary
total_count = all_concepts["combined_results"]["total_count"]

## Step 4: Export to CSV

In [ ]:
# Build DataFrame from all concepts
records = []

for result_key, result in all_concepts.items():
    for concept in result["concepts"]:
        records.append(
            {
                "base_concept_cui": result_key,
                "related_concept_cui": concept["cui"],
                "related_concept_name": concept["name"],
                "source_term": "combined_expansion",
            }
        )

# Create DataFrame
df = pd.DataFrame(records)

# Save to CSV - use relative path
CURRENT_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in locals()
    else os.getcwd()
)
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
output_path = os.path.join(PROJECT_ROOT, "hemochromatosis_expansion_results.csv")
df.to_csv(output_path, index=False)

total_records = len(df)

## Verification

In [ ]:
# Verify file was created
if os.path.exists(output_path):
    df_check = pd.read_csv(output_path)
else:
    pass

In [ ]:
df_check